# 10 · MVP demo: Skin Lesion Triage

**What this notebook does:** starts the Streamlit demo and gives it a temporary public address.
A patient submits a photo and six answers → the image and questionnaire models score the case →
the patient is ranked in a simulated daily queue → the Agent can prepare evidence for the
dermatologist → a Word report is available.

**One source of code:** the notebook always clones the latest `main` of
[IfatDav/skin-lesion-assistant](https://github.com/IfatDav/skin-lesion-assistant).
It does **not** mount Google Drive or copy an MVP zip, local folder, or old Agent code
over the repository. Cell 2 prints the exact Git commit used, so the demo is traceable.

**Before running:**
- Colab Secrets (key icon, left): add `GEMINI_API_KEY`, with **Notebook access** enabled,
  to run the live Agent. The image model and non-LLM fallback can run without an API key.
- Optional: Runtime → Change runtime type → **T4 GPU** (image inference is faster; CPU also works).
- The last cell exposes a **public** demo link. Use synthetic/example cases only;
  **do not upload identifiable patient data or real clinical photographs**.

Run **Runtime → Run all**. Each run recreates the temporary checkout from GitHub,
without writing changes back to GitHub or Drive.


In [ ]:
# Cell 1 · Configuration: GitHub main is the only source of application code
import os, sys, subprocess, shutil, time, tempfile
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/IfatDav/skin-lesion-assistant.git"
BRANCH = "main"
WORK = str(Path("/content" if IN_COLAB else tempfile.gettempdir()) / "skin_triage_app")
PORT = 8501

print("Code source:", REPO_URL, "| branch:", BRANCH)
print("Temporary checkout:", WORK)


In [ ]:
# Cell 2 · Fresh clone from GitHub only; never overlay files from Drive or a local MVP folder
checkout = Path(WORK)
os.chdir(checkout.parent)  # leave WORK before recreating it on a rerun
if checkout.exists():
    # Avoid deleting a directory that is not the checkout made by this notebook.
    if not checkout.is_dir() or not (checkout / ".git").is_dir():
        raise RuntimeError(f"Refusing to replace a non-Git folder: {checkout}")
    existing_origin = subprocess.run(
        ["git", "-C", WORK, "remote", "get-url", "origin"],
        check=True, capture_output=True, text=True
    ).stdout.strip()
    if existing_origin != REPO_URL:
        raise RuntimeError(f"Refusing to replace an unrelated Git checkout: {existing_origin}")
    shutil.rmtree(checkout)

subprocess.run(
    ["git", "clone", "--quiet", "--depth", "1", "--single-branch", "-b", BRANCH, REPO_URL, WORK],
    check=True
)
required = ["app/streamlit_app.py", "app/engine.py", "agent/triage_agent.py"]
missing = [p for p in required if not (checkout / p).is_file()]
if missing:
    raise FileNotFoundError(f"Required files missing in GitHub {BRANCH}: {missing}")

commit = subprocess.check_output(["git", "-C", WORK, "rev-parse", "HEAD"], text=True).strip()
changes = subprocess.check_output(["git", "-C", WORK, "status", "--porcelain"], text=True).strip()
if changes:
    raise RuntimeError("Fresh checkout is unexpectedly modified: " + changes)
print(f"GitHub {BRANCH} cloned → {WORK}")
print("Exact code commit:", commit)
print("Source check: clean checkout; no Drive or local overrides")
os.chdir(WORK)


In [ ]:
# Cell 3 · Install (PyTorch is already in Colab; this adds the rest)
!pip install -q streamlit timm python-docx pydantic "scikit-learn==1.9.1" crewai google-genai litellm chromadb sentence-transformers
print("installed")

In [ ]:
# Cell 4 · The Gemini key: from Colab Secrets, or typed once if Secrets are not reachable
from getpass import getpass
key = None
try:
    from google.colab import userdata
    key = userdata.get("GEMINI_API_KEY")
except Exception as e:   # not added, Notebook access off, or the secret request timed out
    print(f"Colab Secrets did not return the key ({type(e).__name__}).")
    print("Fix it in the key icon on the left (Notebook access ON), or paste the key now:")
if not key:
    key = getpass("GEMINI_API_KEY: ")
os.environ["GEMINI_API_KEY"] = os.environ["GOOGLE_API_KEY"] = key
os.environ["RAG_DIR"] = os.path.join(WORK, "data", "processed", "rag")
os.environ["CREWAI_TRACING_ENABLED"] = "false"     # no "Share this execution trace?" question (20 s wait)
os.environ["CREWAI_DISABLE_TELEMETRY"] = "true"
# os.environ["TRIAGE_MODEL"] = "gemini-3.5-flash"   # optional: a different Gemini model
print("key loaded (not printed)")

In [ ]:
# Cell 5 · The guidance index the agent cites from (NCI PDQ, 724 passages; about a minute)
!python agent/rag.py build

In [ ]:
# Cell 6 · The image model: downloads the weights once and checks they load
!python -c "from src.predict import VisualRiskModel as V; m=V(); print('ConvNeXt ready on', m.device, '| test score:', m.predict(['app/examples/nevus.jpg'])['cancer_score'])"

In [ ]:
# Cell 7 · Rehearsal: one live agent call, so a problem shows here and not in front of the investor
!python -c "import sys; sys.path.insert(0,'app'); import engine as E; from PIL import Image; ans=dict(itch='yes',grew='yes',hurt='no',changed='no',bleed='no',elevation='yes'); r=E.rank_today(0.97, E.questionnaire_score(66,'NECK',ans)); c=E.make_case('NEW-01',r['rank'],r,66,'NECK',ans,'השומה החשיכה בחודש האחרון','app/examples/contradiction.jpg',E.photo_quality(Image.open('app/examples/contradiction.jpg'))); rv,how=E.review(c,'live'); print('engine:', how); print('decision:', rv.decision); print('message:', rv.follow_up_question or rv.retake_instruction)"
# Expected: engine: live agent · decision: ask_patient · a question in Hebrew.
# If it says "live agent unavailable", read the reason in brackets.

In [ ]:
# Cell 8 · Start the app and give it a public address
from IPython.display import HTML, display
subprocess.run("pkill -f 'streamlit run' ; pkill -f cloudflared", shell=True)
app_log = open("/content/streamlit.log", "w")
subprocess.Popen(["streamlit", "run", "app/streamlit_app.py", "--server.port", str(PORT), "--server.headless", "true",
                  "--server.enableCORS", "false", "--server.enableXsrfProtection", "false"],   # uploads work through the tunnel
                 cwd=WORK, env=os.environ.copy(), stdout=app_log, stderr=subprocess.STDOUT)
if not os.path.exists("/content/cloudflared"):
    subprocess.run(["wget", "-q", "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64",
                    "-O", "/content/cloudflared"], check=True)
    os.chmod("/content/cloudflared", 0o755)
tunnel_log = open("/content/tunnel.log", "w")
subprocess.Popen(["/content/cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"], stdout=tunnel_log, stderr=subprocess.STDOUT)

url = None
for _ in range(30):
    time.sleep(2)
    for line in open("/content/tunnel.log"):
        if "trycloudflare.com" in line and "https://" in line:
            url = line[line.index("https://"):].split()[0]
    if url:
        break
if url:
    display(HTML(f'<p style="font-size:22px">The app is live: <a href="{url}" target="_blank">{url}</a></p>'
                 '<p>The address works while this notebook is running.</p>'))
else:
    print("No address yet. Run Cell 9 to see the logs.")

In [ ]:
# Cell 9 · Troubleshooting: the last lines of each log
!tail -n 25 /content/streamlit.log
!tail -n 5 /content/tunnel.log